# By_ZCTA Data Processing Notebook

This notebook extracts and processes ZIP Code Tabulation Area (ZCTA)-level demographic and socioeconomic data from the American Community Survey (ACS). These ZCTA-level indicators will later be used to support more localized analyses and to complement the county-level dataset previously constructed. Since the Medicaid fraud dataset spans only three years (2021–2023), ACS 5-Year Estimates for those same years are used to maintain consistency and reliability across geographic levels.

ZCTAs serve as a finer geographic unit than counties, allowing for more granular exploration of local variation. This increased precision is especially useful for detecting localized socioeconomic patterns relevant to fraud detection and healthcare utilization.

## Key Variables Extracted

### **Demographic Indicators**
- TotalPopulation
- Race and ethnicity percentages:
  - Pct_White
  - Pct_Black
  - Pct_Asian
  - Pct_Hispanic

### **Economic Indicators**
- MedianIncome
- LaborForce
- Unemployed
- UnemploymentRate

### **Poverty Measures**
- PovertyTotal
- PovertyUniverse
- PovertyRate

### **Educational Attainment**
- HighSchoolOrHigher
- BachelorsOrHigher

In [1]:
import requests
import pandas as pd
import numpy as np

## ACS Education Variable Summary

ZCTA-level educational attainment data are drawn from **ACS Table B15003 — Educational Attainment for the Population 25 Years and Over**, representing individuals aged 25+ across a range of education levels:

- **B15003_001** — Total population age 25+

**High school or equivalent:**
- B15003_017 — High school graduate
- B15003_018 — GED or alternative credential
- B15003_019 — Some college, <1 year
- B15003_020 — Some college, ≥1 year
- B15003_021 — Associate’s degree

**Bachelor’s degree or higher:**
- B15003_022 — Bachelor’s degree
- B15003_023 — Master’s degree
- B15003_024 — Professional school degree
- B15003_025 — Doctorate degree

Derived variables:

- **HighSchoolOrHigher** = sum of B15003_017 through B15003_025  
- **BachelorsOrHigher** = sum of B15003_022 through B15003_025


In [2]:
# SETTINGS
API_KEY = "6755c585a386003e31f7c5540c276cd03714bb6d"

# ACS 5-year dataset
DATASET = "acs/acs5"

# Year to pull
YEAR = 2023

# VARIABLES TO REQUEST
variables = {
    "B01003_001E": "TotalPopulation",
    "B19013_001E": "MedianIncome",
    "B17001_002E": "PovertyTotal",
    "B17001_001E": "PovertyUniverse",
    "B02001_002E": "White",
    "B02001_003E": "Black",
    "B02001_005E": "Asian",
    "B03003_003E": "Hispanic",
    "B23025_003E": "LaborForce",
    "B23025_005E": "Unemployed",

    # EDUCATION VARIABLES (same as county version)
    "B15003_017E": "Edu_HS_17",
    "B15003_018E": "Edu_HS_18",
    "B15003_019E": "Edu_HS_19",
    "B15003_020E": "Edu_HS_20",
    "B15003_021E": "Edu_HS_21",
    "B15003_022E": "Edu_BA_22",
    "B15003_023E": "Edu_BA_23",
    "B15003_024E": "Edu_BA_24",
    "B15003_025E": "Edu_BA_25"
}

var_string = ",".join(variables.keys())

## API Configuration and Data Extraction

The notebook retrieves ACS5 data for each year (2021, 2022, and 2023) using a Census API key. For each year, the following steps are performed:

- Construct an API request for demographic, economic, poverty, education, and labor-force indicators at the ZCTA level.
- Convert the JSON API response into a pandas DataFrame.
- Rename variables using clear, human-readable labels.
- Compute derived metrics such as poverty rate, unemployment rate, and education aggregates.
- Perform data quality checks to ensure no rows contain missing, invalid, or placeholder values.

These standardized steps ensure each ZCTA dataset is consistently prepared for comparison across years.

In [3]:
# API CALL FOR ZIP CODES

url = (
    f"https://api.census.gov/data/{YEAR}/{DATASET}"
    f"?get=NAME,{var_string}&for=zip%20code%20tabulation%20area:*"
    f"&key={API_KEY}"
)

response = requests.get(url)
json_data = response.json()

In [4]:
# Convert to DataFrame
df = pd.DataFrame(json_data[1:], columns=json_data[0])

# Convert numeric columns to integers
for col in variables.keys():
    df[col] = pd.to_numeric(df[col], errors="coerce")

# RENAME COLUMNS
df = df.rename(columns=variables)
df = df.rename(columns={"zip code tabulation area": "Zip_Code"})
df.head()

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,LaborForce,...,Edu_HS_17,Edu_HS_18,Edu_HS_19,Edu_HS_20,Edu_HS_21,Edu_BA_22,Edu_BA_23,Edu_BA_24,Edu_BA_25,Zip_Code
0,ZCTA5 00601,16721,18571,10199,16676,13904,314,19,16630,6059,...,3356,403,226,890,1624,1678,408,21,15,00601
1,ZCTA5 00602,37510,21702,17504,37419,13781,520,44,35950,12328,...,6328,928,204,2044,3659,5275,1297,57,116,00602
2,ZCTA5 00603,48317,19243,22683,47655,35550,1572,8,47521,16272,...,10799,948,321,3788,2990,5980,2332,479,427,00603
3,ZCTA5 00606,5435,20226,2984,5435,3697,12,15,5373,1414,...,1304,88,89,345,208,271,182,0,0,00606
4,ZCTA5 00610,25413,23732,11145,25312,6582,525,0,24663,9876,...,4899,540,178,1407,2536,3402,877,246,29,00610


In [5]:
# CREATE DERIVED VARIABLES

# Poverty Rate
df["PovertyRate"] = df["PovertyTotal"] / df["PovertyUniverse"]

# Race Percentages
df["Pct_White"] = df["White"] / df["TotalPopulation"]
df["Pct_Black"] = df["Black"] / df["TotalPopulation"]
df["Pct_Asian"] = df["Asian"] / df["TotalPopulation"]
df["Pct_Hispanic"] = df["Hispanic"] / df["TotalPopulation"]

# Education Levels
df["HighSchoolOrHigher"] = (
    df["Edu_HS_17"] + df["Edu_HS_18"] + df["Edu_HS_19"] +
    df["Edu_HS_20"] + df["Edu_HS_21"] +
    df["Edu_BA_22"] + df["Edu_BA_23"] + df["Edu_BA_24"] + df["Edu_BA_25"]
)

df["BachelorsOrHigher"] = (
    df["Edu_BA_22"] + df["Edu_BA_23"] +
    df["Edu_BA_24"] + df["Edu_BA_25"]
)

# Unemployment Rate
df["UnemploymentRate"] = df["Unemployed"] / df["LaborForce"]

In [6]:
df2023 = df

## Processing Subsequent Years (2021 and 2022)

After completing the full extraction, renaming, transformation, and validation steps for the **2023** dataset, the same workflow is repeated for **2021** and **2022**.

For each of the later years:

- A new ACS API request is issued.
- The JSON response is converted into a DataFrame.
- All variable renaming and derived calculations are applied identically.
- Data quality filtering is repeated.
- The cleaned DataFrame is stored for merging.

This ensures that the structure, variables, and calculations are fully aligned across all three years.

In [7]:
# Year to pull
YEAR = 2022

url = (
    f"https://api.census.gov/data/{YEAR}/{DATASET}"
    f"?get=NAME,{var_string}&for=zip%20code%20tabulation%20area:*"
    f"&key={API_KEY}"
)

response = requests.get(url)
json_data = response.json()

# Convert to DataFrame
df = pd.DataFrame(json_data[1:], columns=json_data[0])

# Convert numeric columns to integers
for col in variables.keys():
    df[col] = pd.to_numeric(df[col], errors="coerce")

# RENAME COLUMNS
df = df.rename(columns=variables)
df = df.rename(columns={"zip code tabulation area": "Zip_Code"})

# CREATE DERIVED VARIABLES

# Poverty Rate
df["PovertyRate"] = df["PovertyTotal"] / df["PovertyUniverse"]

# Race Percentages
df["Pct_White"] = df["White"] / df["TotalPopulation"]
df["Pct_Black"] = df["Black"] / df["TotalPopulation"]
df["Pct_Asian"] = df["Asian"] / df["TotalPopulation"]
df["Pct_Hispanic"] = df["Hispanic"] / df["TotalPopulation"]

# Education Levels
df["HighSchoolOrHigher"] = (
    df["Edu_HS_17"] + df["Edu_HS_18"] + df["Edu_HS_19"] +
    df["Edu_HS_20"] + df["Edu_HS_21"] +
    df["Edu_BA_22"] + df["Edu_BA_23"] + df["Edu_BA_24"] + df["Edu_BA_25"]
)

df["BachelorsOrHigher"] = (
    df["Edu_BA_22"] + df["Edu_BA_23"] +
    df["Edu_BA_24"] + df["Edu_BA_25"]
)

# Unemployment Rate
df["UnemploymentRate"] = df["Unemployed"] / df["LaborForce"]

df.head()

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,LaborForce,...,Edu_BA_25,Zip_Code,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate
0,ZCTA5 00601,16834,17526,10440,16791,14170,348,23,16726,6058,...,23,00601,0.621762,0.841749,0.020672,0.001366,0.993584,8432,2056,0.224166
1,ZCTA5 00602,37642,20260,17768,37510,18479,555,48,35608,12182,...,109,00602,0.473687,0.490914,0.014744,0.001275,0.945965,18904,6126,0.076835
2,ZCTA5 00603,49075,17703,23551,48379,36216,1719,28,48141,16076,...,337,00603,0.486802,0.737972,0.035028,0.000571,0.980968,27474,8884,0.161359
3,ZCTA5 00606,5590,19603,3021,5590,3721,9,0,5552,1449,...,0,00606,0.540429,0.665653,0.001610,0.000000,0.993202,2450,416,0.057281
4,ZCTA5 00610,25542,22796,11597,25405,11182,576,0,24609,9542,...,17,00610,0.456485,0.437789,0.022551,0.000000,0.963472,13826,4213,0.082897


In [8]:
df2022 = df

In [9]:
# Year to pull
YEAR = 2021

url = (
    f"https://api.census.gov/data/{YEAR}/{DATASET}"
    f"?get=NAME,{var_string}&for=zip%20code%20tabulation%20area:*"
    f"&key={API_KEY}"
)

response = requests.get(url)
json_data = response.json()

# Convert to DataFrame
df = pd.DataFrame(json_data[1:], columns=json_data[0])

# Convert numeric columns to integers
for col in variables.keys():
    df[col] = pd.to_numeric(df[col], errors="coerce")

# RENAME COLUMNS
df = df.rename(columns=variables)
df = df.rename(columns={"zip code tabulation area": "Zip_Code"})

# CREATE DERIVED VARIABLES

# Poverty Rate
df["PovertyRate"] = df["PovertyTotal"] / df["PovertyUniverse"]

# Race Percentages
df["Pct_White"] = df["White"] / df["TotalPopulation"]
df["Pct_Black"] = df["Black"] / df["TotalPopulation"]
df["Pct_Asian"] = df["Asian"] / df["TotalPopulation"]
df["Pct_Hispanic"] = df["Hispanic"] / df["TotalPopulation"]

# Education Levels
df["HighSchoolOrHigher"] = (
    df["Edu_HS_17"] + df["Edu_HS_18"] + df["Edu_HS_19"] +
    df["Edu_HS_20"] + df["Edu_HS_21"] +
    df["Edu_BA_22"] + df["Edu_BA_23"] + df["Edu_BA_24"] + df["Edu_BA_25"]
)

df["BachelorsOrHigher"] = (
    df["Edu_BA_22"] + df["Edu_BA_23"] +
    df["Edu_BA_24"] + df["Edu_BA_25"]
)

# Unemployment Rate
df["UnemploymentRate"] = df["Unemployed"] / df["LaborForce"]

df.head()

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,LaborForce,...,Edu_BA_25,Zip_Code,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate
0,ZCTA5 00601,17126,15292,11302,17074,14463,243,2,17038,5566,...,20,00601,0.661942,0.844505,0.014189,0.000117,0.994862,8118,1811,0.250808
1,ZCTA5 00602,37895,18716,17121,37812,23237,688,46,35649,12218,...,173,00602,0.452793,0.613194,0.018155,0.001214,0.940731,18644,6139,0.076854
2,ZCTA5 00603,49136,16789,23617,48441,36497,1681,38,48121,15784,...,282,00603,0.487542,0.742775,0.034211,0.000773,0.979343,26872,8720,0.171820
3,ZCTA5 00606,5751,18835,3139,5751,3319,27,0,5710,1633,...,0,00606,0.545818,0.577117,0.004695,0.000000,0.992871,2493,436,0.069198
4,ZCTA5 00610,26153,21239,11640,26053,15815,642,0,25053,9464,...,114,00610,0.446782,0.604711,0.024548,0.000000,0.957940,13627,3646,0.090342


In [10]:
df2021 = df

## Comparison Across Years

Once datasets for all years (2021–2023) have been processed, the notebook performs the following comparisons:

- Identifies ZCTAs present in multiple years.
- Examines changes in key indicators across time.
- Detects discrepancies or structural inconsistencies.
- Generates a summary table of differences where necessary.

This step ensures that year-to-year variation reflects legitimate updates in ACS estimates rather than formatting or extraction errors.

In [11]:
# --- prepare copies so we don't modify originals
a = df2022.copy()
b = df2021.copy()

# --- detect duplicate NAMEs and disambiguate if necessary
if a['NAME'].duplicated().any() or b['NAME'].duplicated().any():
    # append an occurrence count to duplicate names to make them unique indices
    a['__dup_idx'] = a.groupby('NAME').cumcount().astype(str).radd('_').replace({'_0': ''})
    b['__dup_idx'] = b.groupby('NAME').cumcount().astype(str).radd('_').replace({'_0': ''})
    a['__NAME_unique'] = a['NAME'] + a['__dup_idx']
    b['__NAME_unique'] = b['NAME'] + b['__dup_idx']
    idx_col = '__NAME_unique'
else:
    idx_col = 'NAME'

# --- set index by NAME (or disambiguated name)
a_idx = a.set_index(idx_col)
b_idx = b.set_index(idx_col)

# --- find names present in both datasets
common_names = a_idx.index.intersection(b_idx.index)

# --- restrict to the common subset
a_common = a_idx.loc[common_names].sort_index()
b_common = b_idx.loc[common_names].sort_index()

# --- mask of rows where any column differs
row_diff_mask = (a_common != b_common).any(axis=1)

# --- basic counts
n_total_common = len(common_names)
n_value_diffs = row_diff_mask.sum()

print(f"Names present in both: {n_total_common}")
print(f"Names with value differences: {n_value_diffs}")

# --- side-by-side combined dataframe (values from df and df0)
left = a_common[row_diff_mask].add_suffix('_df')
right = b_common[row_diff_mask].add_suffix('_df0')
combined = pd.concat([left, right], axis=1)

# --- numeric columns difference (df - df0) for numeric fields only
numeric_cols = a_common.select_dtypes(include=[np.number]).columns.intersection(
               b_common.select_dtypes(include=[np.number]).columns)
if len(numeric_cols) > 0:
    diff_numeric = (a_common[numeric_cols] - b_common[numeric_cols]).loc[row_diff_mask]
    diff_numeric = diff_numeric.add_suffix('_diff')
    report = pd.concat([combined, diff_numeric], axis=1)
else:
    report = combined

# --- optional: list which non-equal columns changed per row
cols_changed = (a_common != b_common).loc[row_diff_mask].apply(lambda row: list(row[row].index), axis=1)
report['columns_changed'] = cols_changed

# --- if we used a unique helper column, restore original NAME column for readability
if idx_col != 'NAME':
    # extract original name before any "_index" suffix
    report.insert(0, 'NAME_original', report.index.to_series().str.split('_').str[0])

# --- preview and save option
print("\nPreview of differences (first 10 rows):")
display(report.head(10))   # in Jupyter this will show a nice table

Names present in both: 33774
Names with value differences: 33758

Preview of differences (first 10 rows):


,TotalPopulation_df,MedianIncome_df,PovertyTotal_df,PovertyUniverse_df,White_df,Black_df,Asian_df,Hispanic_df,LaborForce_df,Unemployed_df,...,Edu_BA_25_diff,PovertyRate_diff,Pct_White_diff,Pct_Black_diff,Pct_Asian_diff,Pct_Hispanic_diff,HighSchoolOrHigher_diff,BachelorsOrHigher_diff,UnemploymentRate_diff,columns_changed
NAME,,,,,,,,,,,,,,,,,,,,,
ZCTA5 00601,16834,17526,10440,16791,14170,348,23,16726,6058,1358,...,3,-0.040180,-0.002757,0.006483,0.001250,-0.001277,314,245,-0.026642,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
ZCTA5 00602,37642,20260,17768,37510,18479,555,48,35608,12182,936,...,-64,0.020894,-0.122280,-0.003411,0.000061,0.005234,260,-13,-0.000019,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
ZCTA5 00603,49075,17703,23551,48379,36216,1719,28,48141,16076,2594,...,55,-0.000739,-0.004803,0.000817,-0.000203,0.001625,602,164,-0.010461,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
ZCTA5 00606,5590,19603,3021,5590,3721,9,0,5552,1449,83,...,0,-0.005389,0.088536,-0.003085,0.000000,0.000331,-43,-20,-0.011917,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
ZCTA5 00610,25542,22796,11597,25405,11182,576,0,24609,9542,791,...,-97,0.009703,-0.166922,-0.001997,0.000000,0.005532,199,567,-0.007446,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
ZCTA5 00611,1315,22525,587,1315,765,14,0,1258,565,119,...,2,-0.024384,0.082139,0.000514,0.000000,0.005758,75,31,0.002286,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
ZCTA5 00612,63312,22305,27239,62890,23571,1775,560,62913,22420,4034,...,124,-0.023477,-0.145473,-0.002172,0.000388,0.001203,652,564,-0.004444,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
ZCTA5 00616,9625,23652,4201,9612,3512,626,147,9601,3725,592,...,9,-0.029805,-0.202366,0.001815,0.001038,0.000059,112,6,-0.044225,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
ZCTA5 00617,22573,20328,10418,22559,8496,754,360,22372,8219,1295,...,-1,0.017411,-0.133291,-0.007381,0.001783,-0.002063,572,-261,-0.001015,"[TotalPopulation, MedianIncome, PovertyTotal, ..."


In [12]:
print(df.columns.tolist())

['NAME', 'TotalPopulation', 'MedianIncome', 'PovertyTotal', 'PovertyUniverse', 'White', 'Black', 'Asian', 'Hispanic', 'LaborForce', 'Unemployed', 'Edu_HS_17', 'Edu_HS_18', 'Edu_HS_19', 'Edu_HS_20', 'Edu_HS_21', 'Edu_BA_22', 'Edu_BA_23', 'Edu_BA_24', 'Edu_BA_25', 'Zip_Code', 'PovertyRate', 'Pct_White', 'Pct_Black', 'Pct_Asian', 'Pct_Hispanic', 'HighSchoolOrHigher', 'BachelorsOrHigher', 'UnemploymentRate']


## Merging and Exporting the Final Dataset

After preparing each individual year:

- A `Year` column is added to the datasets for 2021, 2022, and 2023.
- The datasets are vertically combined into a single multi-year panel dataset.
- The merged data is sorted by ZCTA.
- The final cleaned dataset is exported as:

In [13]:
# Add a 'Year' column to each dataframe
df2021['Year'] = 2021
df2022['Year'] = 2022
df2023['Year'] = 2023

# Concatenate them vertically
merged_df = pd.concat([df2021, df2022, df2023], ignore_index=True)

# Sort by NAME
merged_df = merged_df.sort_values(by='NAME').reset_index(drop=True)

# Optional: preview
merged_df.head(10)

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,LaborForce,...,Zip_Code,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate,Year
0,ZCTA5 00601,17126,15292,11302,17074,14463,243,2,17038,5566,...,00601,0.661942,0.844505,0.014189,0.000117,0.994862,8118,1811,0.250808,2021
1,ZCTA5 00601,16721,18571,10199,16676,13904,314,19,16630,6059,...,00601,0.611598,0.831529,0.018779,0.001136,0.994558,8621,2122,0.214722,2023
2,ZCTA5 00601,16834,17526,10440,16791,14170,348,23,16726,6058,...,00601,0.621762,0.841749,0.020672,0.001366,0.993584,8432,2056,0.224166,2022
3,ZCTA5 00602,37895,18716,17121,37812,23237,688,46,35649,12218,...,00602,0.452793,0.613194,0.018155,0.001214,0.940731,18644,6139,0.076854,2021
4,ZCTA5 00602,37510,21702,17504,37419,13781,520,44,35950,12328,...,00602,0.467784,0.367395,0.013863,0.001173,0.958411,19908,6745,0.061324,2023
5,ZCTA5 00602,37642,20260,17768,37510,18479,555,48,35608,12182,...,00602,0.473687,0.490914,0.014744,0.001275,0.945965,18904,6126,0.076835,2022
6,ZCTA5 00603,49136,16789,23617,48441,36497,1681,38,48121,15784,...,00603,0.487542,0.742775,0.034211,0.000773,0.979343,26872,8720,0.171820,2021
7,ZCTA5 00603,48317,19243,22683,47655,35550,1572,8,47521,16272,...,00603,0.475984,0.735766,0.032535,0.000166,0.983525,28064,9218,0.167711,2023
8,ZCTA5 00603,49075,17703,23551,48379,36216,1719,28,48141,16076,...,00603,0.486802,0.737972,0.035028,0.000571,0.980968,27474,8884,0.161359,2022
9,ZCTA5 00606,5751,18835,3139,5751,3319,27,0,5710,1633,...,00606,0.545818,0.577117,0.004695,0.000000,0.992871,2493,436,0.069198,2021


In [14]:
merged_df.shape

(101320, 30)

## Checking randomly

In [15]:
df[df['Zip_Code'].astype(str).str.startswith('976')]

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,LaborForce,...,Zip_Code,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate,Year
32801,ZCTA5 97601,22967,45407,5061,22605,19810,183,443,2918,9891,...,97601,0.223889,0.862542,0.007968,0.019289,0.127052,13941,3598,0.082196,2021
32802,ZCTA5 97603,30950,56549,5126,30404,24775,211,298,4745,13385,...,97603,0.168596,0.800485,0.006817,0.009628,0.153312,19067,4884,0.065372,2021
32803,ZCTA5 97604,0,-666666666,0,0,0,0,0,0,0,...,97604,NaN,NaN,NaN,NaN,NaN,0,0,NaN,2021
32804,ZCTA5 97620,51,-666666666,4,51,51,0,0,0,19,...,97620,0.078431,1.000000,0.000000,0.000000,0.000000,37,15,0.000000,2021
32805,ZCTA5 97621,199,-666666666,0,199,153,0,0,0,79,...,97621,0.000000,0.768844,0.000000,0.000000,0.000000,183,50,0.000000,2021
32806,ZCTA5 97622,310,25417,98,310,246,0,0,0,115,...,97622,0.316129,0.793548,0.000000,0.000000,0.000000,227,44,0.147826,2021
32807,ZCTA5 97623,2677,58611,395,2636,2400,15,0,247,1010,...,97623,0.149848,0.896526,0.005603,0.000000,0.092267,1816,439,0.082178,2021
32808,ZCTA5 97624,3958,49327,864,3946,3010,1,17,227,1361,...,97624,0.218956,0.760485,0.000253,0.004295,0.057352,2564,457,0.126378,2021
32809,ZCTA5 97625,161,72917,12,161,161,0,0,0,96,...,97625,0.074534,1.000000,0.000000,0.000000,0.000000,101,36,0.083333,2021
32810,ZCTA5 97626,112,28182,16,112,112,0,0,0,30,...,97626,0.142857,1.000000,0.000000,0.000000,0.000000,112,30,0.000000,2021


## Data Cleaning and Quality Filtering (By_ZCTA)

After extracting the complete ZCTA-level dataset, several cleaning steps were applied to ensure that the data was valid, consistent, and ready for analysis. These steps addressed missing values, placeholder entries, incorrectly formatted fields, and column organization.

---

### 1. Reordering and Dropping Columns

In [16]:
df = merged_df

In [17]:
df.insert(0, "Zip_Code", df.pop("Zip_Code"))
df.drop("NAME", axis=1, inplace=True)

In [18]:
df[df['Zip_Code'] == '97635']

,Zip_Code,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,LaborForce,...,Edu_BA_25,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate,Year
98446,97635,177,-666666666,44,177,158,0,0,0,56,...,0,0.248588,0.892655,0.0,0.0,0.0,119,41,0.0,2021
98447,97635,345,-666666666,27,345,345,0,0,0,97,...,0,0.078261,1.000000,0.0,0.0,0.0,206,44,0.0,2022
98448,97635,323,-666666666,40,323,323,0,0,0,98,...,0,0.123839,1.000000,0.0,0.0,0.0,198,51,0.0,2023


### 2. Identifying Invalid or Unclean Rows

To ensure the dataset contained only valid ZCTA records, several checks were applied across all columns:

In [19]:
# Detect unclean rows
cond_nan = df.isna().any(axis=1)
cond_empty = df.apply(lambda row: row.astype(str).str.strip().eq('')).any(axis=1)
cond_dash = df.apply(lambda row: row.astype(str).str.strip().eq('-')).any(axis=1)
cond_negative = df.apply(pd.to_numeric, errors="coerce").lt(0).any(axis=1)

# Combine conditions
unclean_rows_df = df[cond_nan | cond_empty | cond_dash | cond_negative]
unclean_rows_df.head()

,Zip_Code,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,LaborForce,...,Edu_BA_25,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate,Year
42,00636,1174,-666666666,765,1174,724,0,0,1174,350,...,26,0.651618,0.616695,0.0,0.0,1.0,476,214,0.0,2021
43,00636,1071,-666666666,841,1071,754,0,0,1071,312,...,0,0.785247,0.704015,0.0,0.0,1.0,472,193,0.0,2023
44,00636,1057,-666666666,800,1057,603,0,0,1057,302,...,21,0.756859,0.570482,0.0,0.0,1.0,420,146,0.0,2022
132,00694,98,-666666666,27,98,0,0,0,98,14,...,0,0.275510,0.000000,0.0,0.0,1.0,75,12,0.0,2022
133,00694,86,-666666666,0,86,28,0,0,86,14,...,0,0.000000,0.325581,0.0,0.0,1.0,64,0,0.0,2021


In [20]:
unclean_rows_df.shape

(9491, 29)

In [21]:
clean_df = df[~(cond_nan | cond_empty | cond_dash | cond_negative)]

In [22]:
clean_df.head()

,Zip_Code,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,LaborForce,...,Edu_BA_25,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate,Year
0,00601,17126,15292,11302,17074,14463,243,2,17038,5566,...,20,0.661942,0.844505,0.014189,0.000117,0.994862,8118,1811,0.250808,2021
1,00601,16721,18571,10199,16676,13904,314,19,16630,6059,...,15,0.611598,0.831529,0.018779,0.001136,0.994558,8621,2122,0.214722,2023
2,00601,16834,17526,10440,16791,14170,348,23,16726,6058,...,23,0.621762,0.841749,0.020672,0.001366,0.993584,8432,2056,0.224166,2022
3,00602,37895,18716,17121,37812,23237,688,46,35649,12218,...,173,0.452793,0.613194,0.018155,0.001214,0.940731,18644,6139,0.076854,2021
4,00602,37510,21702,17504,37419,13781,520,44,35950,12328,...,116,0.467784,0.367395,0.013863,0.001173,0.958411,19908,6745,0.061324,2023


In [23]:
clean_df.shape

(91829, 29)

### **Merged_zcta_2021_2023.csv**

This ZCTA dataset serves as a detailed and localized counterpart to the county-level version and will be used for modeling, visualization, and deeper investigation into how community characteristics relate to Medicaid fraud patterns.


In [24]:
# EXPORT CSV
output_file = "zcta_acs_data_full.csv"
clean_df.to_csv(output_file, index=False)

print(f"\nCSV saved as: {output_file}\n")


CSV saved as: zcta_acs_data_full.csv

